# TabICLv2 Regressor artifact inference — standalone Colab

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook spec:** DIMER Notebook Specification `1.0`  
**Capability:** validate an externally supplied `tabicl-dimer-regressor-v1` serving artifact, reconstruct its in-context serving state, score genuinely new tabular rows, and export point predictions.

**API boundary.** This umbrella repository defines the artifact/serving contract and exposes the installable `tabicl_regressor_pipeline` public reference API. This notebook installs that API at an immutable commit and uses it for artifact/runtime compatibility, serving-object construction, support-context restoration, point prediction, and notebook I/O. `allow_auto_download=False` remains enforced.

**This notebook does not train or fine-tune**, does not perform classification or forecasting, and does not provide calibrated per-prediction uncertainty. `fit(context_X, context_y)` restores the support context required by the in-context model; it is not a gradient update.

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-regressor-pipeline/blob/main/tutorials/tabiclv2_regressor_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL)
[![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl)
[![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

Someone hands you a TabICL serving bundle, produced by the main tutorial or by the DIMER pipeline, and asks for predictions on new rows. Before you load it you want three things: that the archive is internally consistent (and, when you were given its SHA-256 by a trusted producer, that it is the archive they sent), that it carries everything an in-context model needs (checkpoint **and** training context), and that no code runs beyond the pinned `tabicl` estimator.

This notebook performs **no gradient fine-tuning**. TabICL's required `fit(context_X, context_y)` call only registers the in-context support table before prediction.

**By the end of this notebook you will be able to:**
- **Verify** a bundle: safe extraction, manifest format and version, per-file SHA-256 of the checkpoint and the training context (internal consistency), and, only when you supply a trusted whole-ZIP SHA-256, integrity of the archive itself.
- **Reconstruct** the regressor from the bundle alone, with the inference settings the producer recorded.
- **Score** a new CSV, with the producer's categorical encoders applied identically, and download `predictions.csv`.

> **Trust boundary:** load only artifacts you created yourself or obtained from a trusted source. Safe ZIP extraction prevents path traversal; it does not make a PyTorch checkpoint trustworthy. Paste a known ZIP SHA-256 below when available.

## Prerequisites
- A bundle ZIP (`tabiclv2-regressor-artifact.zip` from the main tutorial, or a DIMER serving bundle in the `tabicl-dimer-regressor-v1` format) and, ideally, the SHA-256 printed at export.
- A CSV of new rows with the bundle's feature columns (no label needed).
- Google Colab, or a clean Jupyter/Python 3.13 runtime with release-verified PyTorch 2.11.0 and a writable `/content` workspace; inference is CPU-capable. About two minutes end to end on the measured release runner.


## 1. Install

The notebook installs exact notebook-level package versions and verifies the release-tested PyTorch framework version before any artifact is deserialized. The manifest's recorded TabICL version is checked against this runtime because checkpoint compatibility is version-sensitive.


In [ ]:
%pip -q install -r "https://raw.githubusercontent.com/kurtvalcorza/tabicl-regressor-pipeline/41a4b2b3537da33b90a6d2562a351f3a3995a74e/tutorials/requirements-release.lock"
%pip -q install --no-deps --no-build-isolation "git+https://github.com/kurtvalcorza/tabicl-regressor-pipeline@41a4b2b3537da33b90a6d2562a351f3a3995a74e"

import importlib.metadata
import sys

import torch
from tabicl_regressor_pipeline import (
    condition_regressor,
    create_regressor,
    download_output,
    predict_points,
    read_single_input,
    runtime_identity,
    validate_artifact_runtime,
)

EXPECTED_TORCH_VERSION = "2.11.0"
observed_torch = torch.__version__.split("+")[0]
if observed_torch != EXPECTED_TORCH_VERSION:
    raise RuntimeError(
        f"Supported runtime requires torch=={EXPECTED_TORCH_VERSION}; this runtime has {torch.__version__}."
    )
TABICL_VERSION = "2.1.1"
identity = runtime_identity()
print("Pipeline API revision:", "41a4b2b3537da33b90a6d2562a351f3a3995a74e")
print("Python:", identity["pythonVersion"])
print("TabICL:", identity["tabiclVersion"])
print("PyTorch:", identity["torchVersion"])
print("pandas:", importlib.metadata.version("pandas"))
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("pyarrow:", importlib.metadata.version("pyarrow"))
print("CUDA:", torch.cuda.is_available())


## 2. Upload and verify the artifact ZIP

In order: the archive's SHA-256 is computed and, if you pasted `EXPECTED_ZIP_SHA256`, compared (**this is the only check that establishes the archive is the one your producer sent**; without it, a bundle whose files and manifest were altered together still passes); every member is checked for absolute paths, `..` segments and symlinks and for escaping the extraction folder; the archive is extracted; exactly one `artifact.json` must exist; its `artifactFormat` and `tabiclVersion` must match this notebook; the checkpoint and training-context paths named in the manifest are resolved *inside* the bundle root; and each file's SHA-256 must equal the digest recorded in the manifest.

**What to look for:** `✓ Artifact structure and digests verified`, which means the bundle is well-formed and internally consistent. Any other outcome means stop and ask the producer.


In [ ]:
import csv
import hashlib
import io
import json
import shutil
import stat
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

EXPECTED_ZIP_SHA256 = ""  # @param {type:"string"}
MAX_EXPANDED_BYTES = 1024 * 1024 * 1024


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract_zip(zip_path, dest):
    """Extract only after every member passed the path and symlink checks."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        expanded_bytes = 0
        for info in archive.infolist():
            if "\\" in info.filename:
                raise ValueError(f"Ambiguous backslash ZIP member: {info.filename}")
            expanded_bytes += info.file_size
            if expanded_bytes > MAX_EXPANDED_BYTES:
                raise ValueError("Artifact exceeds 1 GiB expanded-size limit")
            name = info.filename
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            target = (dest / Path(name)).resolve()
            if root != target and root not in target.parents:
                raise ValueError("ZIP member escapes destination")
        archive.extractall(dest)


def manifest_member_path(root, value, field):
    """Resolve a manifest path inside the bundle root, refusing absolute paths and traversal."""
    rel = Path(str(value))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")
    root_resolved = Path(root).resolve()
    target = (root_resolved / rel).resolve()
    if root_resolved != target and root_resolved not in target.parents:
        raise ValueError(f"{field} path escapes artifact root: {value!r}")
    return target


name, payload = read_single_input(env_var="DIMER_ARTIFACT_PATH", label="artifact ZIP")
zip_path = Path("/content") / Path(name).name
zip_path.write_bytes(payload)
observed = sha256_file(zip_path)
print("ZIP SHA-256:", observed)
if EXPECTED_ZIP_SHA256:
    expected = EXPECTED_ZIP_SHA256.strip().lower()
    if len(expected) != 64 or any(character not in "0123456789abcdef" for character in expected):
        raise ValueError("EXPECTED_ZIP_SHA256 must be 64 hex chars")
    if observed != expected:
        raise RuntimeError("Artifact ZIP SHA-256 mismatch")

extract_dir = Path("/content/tabiclv2-regressor-artifact")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
safe_extract_zip(zip_path, extract_dir)

matches = list(extract_dir.rglob("artifact.json"))
if len(matches) != 1:
    raise ValueError("Expected exactly one artifact.json")
root = matches[0].parent
manifest = json.loads(matches[0].read_text())
if manifest.get("artifactFormat") != "tabicl-dimer-regressor-v1":
    raise ValueError(f"Unsupported artifactFormat: {manifest.get('artifactFormat')}")
if manifest.get("tabiclVersion") != TABICL_VERSION:
    raise ValueError("Artifact TabICL version does not match notebook pin")

compatibility = validate_artifact_runtime(
    manifest, expected_tabicl_version=TABICL_VERSION, expected_torch_version=EXPECTED_TORCH_VERSION
)
print("Artifact/runtime provenance and compatibility:")
print(json.dumps(compatibility, indent=2, sort_keys=True))

ckpt = manifest_member_path(root, manifest["checkpoint"], "checkpoint")
context_path = manifest_member_path(root, manifest["trainingContext"], "trainingContext")
payload_files = manifest.get("payloadFiles")
if payload_files is not None:
    expected_files = {"artifact.json", *payload_files}
    actual_files = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}
    if actual_files != expected_files:
        raise RuntimeError(f"Unexpected or missing artifact files: {sorted(actual_files ^ expected_files)}")
else:
    print("⚠ Legacy manifest has no payloadFiles allowlist; unexpected-file rejection cannot be proven.")
if manifest.get("sizes"):
    if ckpt.stat().st_size != manifest["sizes"]["checkpoint"]:
        raise RuntimeError("Checkpoint size mismatch")
    if context_path.stat().st_size != manifest["sizes"]["trainingContext"]:
        raise RuntimeError("Training-context size mismatch")
else:
    print("⚠ Legacy manifest has no recorded payload sizes; digest checks still apply.")
if sha256_file(ckpt) != manifest["digests"]["checkpointSha256"]:
    raise RuntimeError("Checkpoint digest mismatch")
if sha256_file(context_path) != manifest["digests"]["trainingContextSha256"]:
    raise RuntimeError("Training-context digest mismatch")
print("✓ Artifact structure and digests verified")
print(f"  mode={manifest.get('mode')} selection={manifest.get('selectionBasis')} base={manifest.get('baseCheckpoint')} @ {str(manifest.get('baseModelRevision'))[:12]}")


## 3. Reconstruct the in-context regressor

The bundle's `inference` block records how the producer ran the model: ensemble size, random seed, and the categorical encoders fitted on the training split. The regressor is built from the bundled checkpoint with `allow_auto_download=False` (nothing may be fetched behind your back) and then given the training context with `fit`, which registers those rows as its prompt and trains nothing.

Before the notebook asks for inference data, the reconstruction cell prints the exact ordered feature schema carried by the artifact. This is the schema the producer used at serving time; it is not inferred from the uploaded inference CSV.


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import platform
print({"python": platform.python_version(), "torch": torch.__version__, "device": DEVICE})
context = pd.read_parquet(context_path)
FEATURE_COLUMNS = manifest["featureColumns"]
TARGET_COLUMN = manifest["targetColumn"]
inference = manifest["inference"]
model = create_regressor(
    model_path=str(ckpt), allow_auto_download=False,
    n_estimators=inference["nEstimators"], random_state=inference["randomState"], device=DEVICE)
condition_regressor(model, context[FEATURE_COLUMNS], context[TARGET_COLUMN])
print(f"✓ Loaded with {len(context)} context rows and {len(FEATURE_COLUMNS)} features")
print("Required feature columns:", FEATURE_COLUMNS)
print("Target recorded by artifact:", TARGET_COLUMN)


## 4. Upload rows and predict

Upload one CSV with the bundle's feature columns. Order does not matter and extra columns are preserved in the output; duplicate header names, missing features, or a pre-existing `prediction` column stop the run. Categoricals are encoded with the producer's map, and values that map to "unknown" are counted in a warning, because a batch full of unseen categories is usually a schema drift you want to know about.

The output adds one scalar `prediction` column in the target's units. There is no uncertainty column; if a decision depends on how wrong an estimate might be, measure the bundle's MAE on rows you know the answer for. That MAE is the typical absolute error observed on held-out data, not a per-prediction uncertainty interval.

**BYOD privacy boundary.** The uploaded CSV is read and scored inside the current notebook runtime; this notebook does not send rows to an external inference service. Google Colab is a hosted environment, so do not upload confidential, restricted, personal, or otherwise sensitive data unless your authorization and data-handling rules permit processing it there. The exact required feature names were printed in Step 3 before this upload prompt.


In [ ]:
def raw_header(payload):
    """First non-empty CSV row, read before pandas can rename duplicate names."""
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(cell.strip() for cell in row):
            return row
    raise ValueError("CSV has no header")


def read_inference_csv(payload, feature_columns):
    header = raw_header(payload)
    seen, dupes = set(), []
    for column in header:
        if column in seen and column not in dupes:
            dupes.append(column)
        seen.add(column)
    if dupes:
        raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")
    frame = pd.read_csv(io.BytesIO(payload))
    if "prediction" in frame.columns:
        raise ValueError("Inference CSV already contains a 'prediction' column")
    missing = [column for column in feature_columns if column not in frame.columns]
    if missing:
        raise ValueError(f"Inference CSV missing features: {missing}")
    return frame


def apply_encoder(frame, encoders, label="data"):
    """Apply the producer's ordinal maps; unseen or missing values get the extra 'unknown' code."""
    out = frame.copy()
    for column, categories in encoders.items():
        lookup = {category: index for index, category in enumerate(categories)}
        unknown = len(categories)
        encoded, unseen = [], 0
        for value in out[column]:
            if pd.isna(value):
                encoded.append(unknown)
                continue
            key = str(value)
            if key not in lookup:
                unseen += 1
            encoded.append(lookup.get(key, unknown))
        out[column] = encoded
        if unseen:
            print(f"⚠ {label}: {unseen} unseen categorical value(s) in {column!r} encoded as unknown.")
    return out


print("Expected inference schema:", FEATURE_COLUMNS)
print("Upload stays in this notebook runtime; no external inference service is called.")
_, payload = read_single_input(env_var="DIMER_INFERENCE_CSV_PATH", label="inference CSV")
rows = read_inference_csv(payload, FEATURE_COLUMNS)
X = apply_encoder(rows[FEATURE_COLUMNS], inference.get("categoricalEncoders", {}), "inference CSV")
out = rows.copy()
out["prediction"] = predict_points(model, X)
output_path = Path("/content/tabiclv2_regressor_predictions.csv")
out.to_csv(output_path, index=False)
print(f"✓ Wrote {len(out)} predictions to {output_path}")
download_output(output_path)


## What a successful run proves — and what it does not

A successful run proves that the supplied archive passed this notebook's path/symlink/expanded-size checks, that its declared payload matches the manifest digests (and sizes/allowlist when provided), that the recorded TabICL runtime contract could be reconstructed without automatic model fallback, that the persisted support context was restored, and that the reconstructed model produced a machine-readable `prediction` column for new rows.

It does **not** prove who authored the archive unless you independently trust and verify the whole-archive digest; internal manifest consistency is not sender authenticity. It also does not establish predictive quality, fairness, calibration, robustness, or production fitness on your deployment population. Point predictions carry no per-row uncertainty interval. Validate the artifact on representative labelled data and apply domain-specific risk controls before consequential use.

## When a check fails

| Error | Meaning | What to do |
|---|---|---|
| `Artifact ZIP SHA-256 mismatch` | not the archive whose digest you were given | get it again; never edit the expected digest |
| `Unsafe ZIP member` / `ZIP member escapes destination` | the archive tries to write outside its folder | reject it; that is what a malicious archive looks like |
| `Unsupported artifactFormat` | a classifier bundle, or not a TabICL bundle | use the matching notebook |
| `Artifact TabICL version does not match notebook pin` | produced with another `tabicl` release | install that release, or re-export |
| `Checkpoint digest mismatch` / `Training-context digest mismatch` | a file inside the bundle differs from its manifest entry | reject the bundle |
| `Inference CSV missing features` | schema mismatch | add the listed columns with the training names |
| many `unseen categorical value(s)` | the new data uses categories the producer never saw | check for a schema change before trusting the predictions |

## AI provenance

This inference tutorial was developed with substantial AI assistance under maintainer direction and review: original build by **GPT-5.6 Sol High**, via **OpenAI / ChatGPT**, under Agent Relay role **Builder**; content revision by **Claude Fable 5.1**, via **Anthropic / Claude Code**, under Agent Relay role **Reviewer and Builder**; review refinement and companion discoverability by **Gemini 3.8 Flash High**, via **Google DeepMind / Antigravity**, under Agent Relay role **Builder**. Provenance only; not independent sign-off.
